# CryptoRisk & Trend SBC
**Sistema Basado en Conocimiento para evaluación de riesgo y decisiones de *swing trading* en criptomonedas**

Proyecto Tercer Corte · IAIA · Grupo MORADO: Santiago Andres Cordoba Torres · Cesar Morales

> Fines académicos. No constituye asesoría financiera.

## 1. Dominio y justificación
El **análisis técnico** de criptomonedas combina tendencia (EMA de 200 periodos), impulso (RSI), volumen,
soportes históricos y contexto macro/regulatorio. Los operadores de *swing trading* ya razonan con reglas del
tipo «si ocurre A y B, entonces C», por lo que el dominio es adecuado para un SBC con **cláusulas definidas de Horn**:

- **Jerarquía natural:** datos de mercado → estados intermedios → recomendación (`recomendar_compra` / `recomendar_caja`).
- **Incertidumbre:** un indicador puede no estar disponible; la lógica trivalente (`YES`/`NO`/`UNKNOWN`) lo representa y permite rutas alternativas.
- **Trazabilidad:** cada conclusión se rastrea hasta los hechos que la originaron, requisito para decisiones con riesgo de capital.
- **Viabilidad:** 11 proposiciones y 7 reglas son abarcables, verificables por un experto y ejecutables en el motor del laboratorio.

El marco teórico (lógica proposicional, cláusulas de Horn y encadenamiento) sigue a Russell & Norvig (2021, cap. 7).

In [1]:
# El motor vive en motor/motor.py (adaptado del laboratorio 05).
# Importa TValue, TAtom, TRule, TKnowledgeBase, TFactBase, forward, backward, explain, why y los subsistemas.
from motor import *

## 2. Modelado formal

### 2.1 Proposiciones atómicas (11)
| symbol | meaning | nivel |
|---|---|---|
| precio_sobre_ema200 | el precio está por encima de la EMA de 200 periodos | entrada |
| rsi_sobreventa | el RSI está por debajo de 30 | entrada |
| rsi_sobrecompra | el RSI está por encima de 70 | entrada |
| volumen_creciente | el volumen de las últimas sesiones va en aumento | entrada |
| soporte_tecnico_clave | el precio rebotó en un soporte histórico importante | entrada |
| noticias_macro_negativas | hay noticias macro o regulatorias adversas | entrada |
| tendencia_saludable | el mercado mantiene una estructura de precios favorable | intermedio |
| alerta_correccion | hay indicios técnicos de posible retroceso | intermedio |
| oportunidad_compra | las condiciones sugieren un buen punto de entrada | intermedio |
| recomendar_compra | abrir posición de compra (riesgo moderado/bajo) | meta |
| recomendar_caja | mantener liquidez / evitar operar (alto riesgo) | meta |

### 2.2 Reglas (cláusulas de Horn)
- R1: precio_sobre_ema200 ∧ soporte_tecnico_clave ⇒ tendencia_saludable
- R2: rsi_sobrecompra ∧ noticias_macro_negativas ⇒ alerta_correccion
- R3: rsi_sobreventa ∧ volumen_creciente ⇒ oportunidad_compra
- R4: rsi_sobreventa ∧ soporte_tecnico_clave ⇒ oportunidad_compra *(respaldo si el volumen es UNKNOWN)*
- R5: tendencia_saludable ∧ oportunidad_compra ⇒ recomendar_compra
- R6a: alerta_correccion ⇒ recomendar_caja
- R6b: noticias_macro_negativas ⇒ recomendar_caja

**¿Por qué se divide R6?** Una cláusula definida de Horn tiene **un único literal positivo** (el consecuente) y su
antecedente es solo una conjunción. La fórmula `alerta_correccion ∨ noticias_macro_negativas ⇒ recomendar_caja`
lleva una disyunción en el antecedente, por lo que es equivalente a dos cláusulas independientes
(`A ⇒ C` y `B ⇒ C`): R6a y R6b. Lo mismo ocurre con R3/R4, que son dos vías alternativas hacia `oportunidad_compra`.

## 3. Valores YES / NO / UNKNOWN
- **Átomo ausente de la FB:** el motor aún no lo ha considerado; `backward` lo preguntará (si es un dato de entrada).
- **Átomo con `UNKNOWN` en la FB:** es una *memoria*: ya se intentó conocerlo y no se sabe; **no se vuelve a preguntar**.
- En `applies`, un antecedente `NO` o `UNKNOWN` impide disparar la regla (un `NO` prevalece: la regla ya no podrá aplicarse).
- **R4 evita el bloqueo:** si `volumen_creciente = UNKNOWN`, R3 no puede aplicarse, pero R4 llega a la misma
  `oportunidad_compra` usando el soporte técnico en lugar del volumen. Sin R4 el sistema quedaría sin conclusión.
- Las reglas de Horn no tienen negación: `NO` solo sirve para bloquear reglas, y `backward` devuelve `UNKNOWN` (no `NO`)
  cuando ninguna regla logra probar la meta.

## 4. Carga de la KB con el Subsistema de Desarrollo
Toda la base de conocimiento se construye **a través de `DevelopmentSubsystem`**, como lo haría el experto:
`dev.add_atom()` para cada proposición y `dev.add_rule()` para cada regla. Primero se registran los 11 átomos y
después las 7 reglas, porque la KB exige que todo átomo de una regla exista antes.

Para que el notebook sea reproducible con *Restart & Run All*, las respuestas que el experto escribiría en `input()`
se reproducen con `respuestas([...])` y se imprimen tal como se teclean. El menú real e interactivo es `dev.run()`.

In [2]:
import builtins
from contextlib import contextmanager

@contextmanager
def respuestas(lista):
    """Sustituye input() por respuestas guiadas y las muestra como si se escribieran."""
    pendientes = iter(lista)
    original = builtins.input
    def falso(prompt=""):
        r = next(pendientes)
        print(f"{prompt}{r}")
        return r
    builtins.input = falso
    try:
        yield
    finally:
        builtins.input = original

kb = TKnowledgeBase()
dev = DevelopmentSubsystem(kb)

# 1) Las 11 proposiciones (símbolo, significado), registradas con el subsistema.
definiciones = [
    ("precio_sobre_ema200",      "el precio está por encima de la EMA de 200 periodos"),
    ("rsi_sobreventa",           "el RSI está por debajo de 30"),
    ("rsi_sobrecompra",          "el RSI está por encima de 70"),
    ("volumen_creciente",        "el volumen de las últimas sesiones va en aumento"),
    ("soporte_tecnico_clave",    "el precio rebotó en un soporte histórico importante"),
    ("noticias_macro_negativas", "hay noticias macro o regulatorias adversas"),
    ("tendencia_saludable",      "el mercado mantiene una estructura de precios favorable"),
    ("alerta_correccion",        "hay indicios técnicos de posible retroceso"),
    ("oportunidad_compra",       "las condiciones sugieren un buen punto de entrada"),
    ("recomendar_compra",        "abrir posición de compra (riesgo moderado/bajo)"),
    ("recomendar_caja",          "mantener liquidez / evitar operar (alto riesgo)"),
]
for simbolo, significado in definiciones:
    with respuestas([simbolo, significado]):
        dev.add_atom()

# Atajo para recuperar átomos por símbolo en las celdas siguientes.
A = lambda s: kb.atom(s)


Símbolo (identificador): precio_sobre_ema200
Significado (lenguaje natural): el precio está por encima de la EMA de 200 periodos
Proposición 'precio_sobre_ema200' registrada.
Símbolo (identificador): rsi_sobreventa
Significado (lenguaje natural): el RSI está por debajo de 30
Proposición 'rsi_sobreventa' registrada.
Símbolo (identificador): rsi_sobrecompra
Significado (lenguaje natural): el RSI está por encima de 70
Proposición 'rsi_sobrecompra' registrada.
Símbolo (identificador): volumen_creciente
Significado (lenguaje natural): el volumen de las últimas sesiones va en aumento
Proposición 'volumen_creciente' registrada.
Símbolo (identificador): soporte_tecnico_clave
Significado (lenguaje natural): el precio rebotó en un soporte histórico importante
Proposición 'soporte_tecnico_clave' registrada.
Símbolo (identificador): noticias_macro_negativas
Significado (lenguaje natural): hay noticias macro o regulatorias adversas
Proposición 'noticias_macro_negativas' registrada.
Símbolo (identif

In [3]:
# 2) Las 7 reglas de Horn (nombre, antecedentes separados por coma, consecuente), con el subsistema.
reglas = [
    ("R1",  "precio_sobre_ema200, soporte_tecnico_clave", "tendencia_saludable"),
    ("R2",  "rsi_sobrecompra, noticias_macro_negativas",  "alerta_correccion"),
    ("R3",  "rsi_sobreventa, volumen_creciente",          "oportunidad_compra"),
    ("R4",  "rsi_sobreventa, soporte_tecnico_clave",      "oportunidad_compra"),
    ("R5",  "tendencia_saludable, oportunidad_compra",    "recomendar_compra"),
    ("R6a", "alerta_correccion",                          "recomendar_caja"),
    ("R6b", "noticias_macro_negativas",                   "recomendar_caja"),
]
for nombre, antecedentes, consecuente in reglas:
    with respuestas([nombre, antecedentes, consecuente]):
        dev.add_rule()


Nombre de la regla: R1
Antecedentes (símbolos separados por coma): precio_sobre_ema200, soporte_tecnico_clave
Consecuente (símbolo): tendencia_saludable
Regla 'R1' registrada.
Nombre de la regla: R2
Antecedentes (símbolos separados por coma): rsi_sobrecompra, noticias_macro_negativas
Consecuente (símbolo): alerta_correccion
Regla 'R2' registrada.
Nombre de la regla: R3
Antecedentes (símbolos separados por coma): rsi_sobreventa, volumen_creciente
Consecuente (símbolo): oportunidad_compra
Regla 'R3' registrada.
Nombre de la regla: R4
Antecedentes (símbolos separados por coma): rsi_sobreventa, soporte_tecnico_clave
Consecuente (símbolo): oportunidad_compra
Regla 'R4' registrada.
Nombre de la regla: R5
Antecedentes (símbolos separados por coma): tendencia_saludable, oportunidad_compra
Consecuente (símbolo): recomendar_compra
Regla 'R5' registrada.
Nombre de la regla: R6a
Antecedentes (símbolos separados por coma): alerta_correccion
Consecuente (símbolo): recomendar_caja
Regla 'R6a' registr

In [4]:
# Validación de integridad: una regla con un átomo NO registrado debe rechazarse.
fantasma = TAtom("macd_cruce_alcista", "el MACD cruza al alza")   # nunca se agregó a la KB
try:
    kb.add(TRule("R_invalida", [fantasma], A("recomendar_compra")))
except ValueError as error:
    print("ValueError capturado:", error)

# Lo mismo a través del menú: el símbolo inexistente se detecta antes de intentar registrar.
with respuestas(["R_invalida", "macd_cruce_alcista", "recomendar_compra"]):
    dev.add_rule()

print()
dev.show()

ValueError capturado: Proposiciones no registradas: macd_cruce_alcista
Nombre de la regla: R_invalida
Antecedentes (símbolos separados por coma): macd_cruce_alcista
Consecuente (símbolo): recomendar_compra
Regla inválida. Proposiciones no registradas: macd_cruce_alcista.

Proposiciones (11):
  - alerta_correccion: hay indicios técnicos de posible retroceso
  - noticias_macro_negativas: hay noticias macro o regulatorias adversas
  - oportunidad_compra: las condiciones sugieren un buen punto de entrada
  - precio_sobre_ema200: el precio está por encima de la EMA de 200 periodos
  - recomendar_caja: mantener liquidez / evitar operar (alto riesgo)
  - recomendar_compra: abrir posición de compra (riesgo moderado/bajo)
  - rsi_sobrecompra: el RSI está por encima de 70
  - rsi_sobreventa: el RSI está por debajo de 30
  - soporte_tecnico_clave: el precio rebotó en un soporte histórico importante
  - tendencia_saludable: el mercado mantiene una estructura de precios favorable
  - volumen_crecie

## 5. Forward chaining
Para no repetir código, `nuevo_caso` construye una FB a partir de un diccionario `símbolo → valor` (origen `user`)
y `mostrar_fb` imprime cada hecho con su valor y origen.

In [5]:
def nuevo_caso(datos):
    """Crea una FB con los hechos aportados por el usuario."""
    fb = TFactBase()
    for simbolo, valor in datos.items():
        fb.set(A(simbolo), valor, "user")
    return fb

def mostrar_fb(fb):
    """Imprime la FB reutilizando el subsistema de consulta (sin interacción)."""
    QuerySubsystem(kb, fb).show()

YES, NO, UNKNOWN = TValue.YES, TValue.NO, TValue.UNKNOWN

### Escenario 1 — Mercado alcista
FB inicial (`user`): `precio_sobre_ema200=YES`, `soporte_tecnico_clave=YES`, `rsi_sobreventa=YES`, `volumen_creciente=YES`.

Esperado: R1 → tendencia_saludable; R3 → oportunidad_compra; R5 → **recomendar_compra**.

In [6]:
fb1 = nuevo_caso({
    "precio_sobre_ema200": YES, "soporte_tecnico_clave": YES,
    "rsi_sobreventa": YES, "volumen_creciente": YES,
})
forward(kb, fb1)
mostrar_fb(fb1)
print()
print(explain(kb, fb1, A("recomendar_compra")))

  precio_sobre_ema200 = YES (user)
  soporte_tecnico_clave = YES (user)
  rsi_sobreventa = YES (user)
  volumen_creciente = YES (user)
  tendencia_saludable = YES (R1)
  oportunidad_compra = YES (R3)
  recomendar_compra = YES (R5)

recomendar_compra = YES  <- R5
  tendencia_saludable = YES  <- R1
    precio_sobre_ema200 = YES  <- user
    soporte_tecnico_clave = YES  <- user
  oportunidad_compra = YES  <- R3
    rsi_sobreventa = YES  <- user
    volumen_creciente = YES  <- user


### Escenario 2 — Incertidumbre: `volumen_creciente = UNKNOWN`
FB inicial: `precio_sobre_ema200=YES`, `soporte_tecnico_clave=YES`, `rsi_sobreventa=YES`, `volumen_creciente=UNKNOWN (user)`.

Esperado: R3 queda bloqueada; **R4** deduce oportunidad_compra; R1 y R5 → recomendar_compra.

In [7]:
fb2 = nuevo_caso({
    "precio_sobre_ema200": YES, "soporte_tecnico_clave": YES,
    "rsi_sobreventa": YES, "volumen_creciente": UNKNOWN,
})
forward(kb, fb2)
mostrar_fb(fb2)
print()
print(explain(kb, fb2, A("recomendar_compra")))

assert fb2.source(A("oportunidad_compra")) == "R4"   # la vía primaria R3 está bloqueada

  precio_sobre_ema200 = YES (user)
  soporte_tecnico_clave = YES (user)
  rsi_sobreventa = YES (user)
  volumen_creciente = UNKNOWN (user)
  tendencia_saludable = YES (R1)
  oportunidad_compra = YES (R4)
  recomendar_compra = YES (R5)

recomendar_compra = YES  <- R5
  tendencia_saludable = YES  <- R1
    precio_sobre_ema200 = YES  <- user
    soporte_tecnico_clave = YES  <- user
  oportunidad_compra = YES  <- R4
    rsi_sobreventa = YES  <- user
    soporte_tecnico_clave = YES  <- user


**Comentario:** `oportunidad_compra` proviene de **R4** y no de R3, porque R3 exige `volumen_creciente = YES` y aquí es
`UNKNOWN`. Sin R4 el sistema no podría concluir `recomendar_compra` aun teniendo evidencia técnica suficiente
(sobreventa en un soporte clave); R4 es redundancia deliberada para tolerar datos faltantes.

### Escenario 3 — Mercado bajista
FB inicial: `precio_sobre_ema200=NO`, `rsi_sobrecompra=YES`, `noticias_macro_negativas=YES`.

Esperado: R2 → alerta_correccion; R6a → **recomendar_caja** (R6b también es aplicable).

In [8]:
fb3 = nuevo_caso({
    "precio_sobre_ema200": NO, "rsi_sobrecompra": YES, "noticias_macro_negativas": YES,
})
forward(kb, fb3)
mostrar_fb(fb3)
print()
print(explain(kb, fb3, A("recomendar_caja")))

print("\n¿Se dedujo recomendar_compra?", A("recomendar_compra") in fb3.facts)
estado_r1 = applies(kb, fb3, kb.obtain("R1"))
print("Estado de R1:", "NO" if estado_r1 is NO else "YES" if estado_r1 is YES else "UNKNOWN")

  precio_sobre_ema200 = NO (user)
  rsi_sobrecompra = YES (user)
  noticias_macro_negativas = YES (user)
  alerta_correccion = YES (R2)
  recomendar_caja = YES (R6a)

recomendar_caja = YES  <- R6a
  alerta_correccion = YES  <- R2
    rsi_sobrecompra = YES  <- user
    noticias_macro_negativas = YES  <- user

¿Se dedujo recomendar_compra? False
Estado de R1: NO


**Comentario:** `recomendar_compra` **no** se deduce: R1 tiene el antecedente `precio_sobre_ema200 = NO`, por lo que
`applies` devuelve `NO` (la regla ya nunca podrá dispararse) y, sin `tendencia_saludable`, R5 tampoco aplica.
Como `recomendar_caja` tiene su origen en R6a (que se evalúa antes que R6b) se conserva ese primer origen.

## 6. Backward chaining (interactivo)

### Escenario 4 — Meta `recomendar_compra` con FB vacía
El motor recorre R5 → R1 / R3 / R4 y **pregunta solo por átomos de entrada** (los que ninguna regla concluye).
Antes de cada pregunta imprime su **¿Por qué?**.

Orden esperado de preguntas: `precio_sobre_ema200`, `soporte_tecnico_clave`, `rsi_sobreventa`, `volumen_creciente`.

Para ejecutarlo de forma manual ponga `INTERACTIVO = True`; por defecto se reproducen respuestas guiadas
(recorrido A: todo `YES`; recorrido B: `volumen_creciente = UNKNOWN`, que obliga a usar R4).

In [9]:
INTERACTIVO = False   # True -> responde usted con input(); False -> respuestas guiadas reproducibles

def correr_backward(respuestas_guiadas):
    fb = TFactBase()
    meta = A("recomendar_compra")
    if INTERACTIVO:
        valor, fb = backward(kb, fb, meta)
    else:
        with respuestas(respuestas_guiadas):
            valor, fb = backward(kb, fb, meta)
    print(f"\nrecomendar_compra = {'YES' if valor is YES else 'UNKNOWN'}\n")
    mostrar_fb(fb)
    print()
    print(explain(kb, fb, meta))
    return fb

print("=== Recorrido A: todo YES ===")
fb4a = correr_backward(["YES", "YES", "YES", "YES"])

=== Recorrido A: todo YES ===
  ¿Por qué? Pregunto «el precio está por encima de la EMA de 200 periodos» porque es necesaria para R1 (concluye tendencia_saludable) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
el precio está por encima de la EMA de 200 periodos (YES/NO/UNKNOWN): YES
  ¿Por qué? Pregunto «el precio rebotó en un soporte histórico importante» porque es necesaria para R1 (concluye tendencia_saludable) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
el precio rebotó en un soporte histórico importante (YES/NO/UNKNOWN): YES
  ¿Por qué? Pregunto «el RSI está por debajo de 30» porque es necesaria para R3 (concluye oportunidad_compra) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
el RSI está por debajo de 30 (YES/NO/UNKNOWN): YES
  ¿Por qué? Pregunto «el volumen de las últimas sesiones va en aumento» porque es necesaria para R3 (concluye oportunidad_compra) -> R5 (concluye recomendar_compra), qu

In [10]:
print("=== Recorrido B: volumen UNKNOWN (se usa R4) ===")
fb4b = correr_backward(["YES", "YES", "YES", "UNKNOWN"])

=== Recorrido B: volumen UNKNOWN (se usa R4) ===
  ¿Por qué? Pregunto «el precio está por encima de la EMA de 200 periodos» porque es necesaria para R1 (concluye tendencia_saludable) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
el precio está por encima de la EMA de 200 periodos (YES/NO/UNKNOWN): YES
  ¿Por qué? Pregunto «el precio rebotó en un soporte histórico importante» porque es necesaria para R1 (concluye tendencia_saludable) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
el precio rebotó en un soporte histórico importante (YES/NO/UNKNOWN): YES
  ¿Por qué? Pregunto «el RSI está por debajo de 30» porque es necesaria para R3 (concluye oportunidad_compra) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
el RSI está por debajo de 30 (YES/NO/UNKNOWN): YES
  ¿Por qué? Pregunto «el volumen de las últimas sesiones va en aumento» porque es necesaria para R3 (concluye oportunidad_compra) -> R5 (concluye rec

**Observaciones:** en ambos recorridos el orden de preguntas es el esperado, y `backward` no pregunta por
`rsi_sobrecompra` ni `noticias_macro_negativas` porque ninguna regla que lleve a `recomendar_compra` las usa.
Si el usuario respondiera `NO` a `precio_sobre_ema200`, el cortocircuito de `all(...)` evitaría preguntar
`soporte_tecnico_clave` para R1.

## 7. Explicabilidad (cómo y por qué)

In [11]:
print("CÓMO — Escenario 1 (compra por R1 + R3):")
print(explain(kb, fb1, A("recomendar_compra")))

print("\nCÓMO — Escenario 2 (compra por R1 + R4):")
print(explain(kb, fb2, A("recomendar_compra")))

print("\nCÓMO — Escenario 3 (caja por R2 → R6a):")
print(explain(kb, fb3, A("recomendar_caja")))

CÓMO — Escenario 1 (compra por R1 + R3):
recomendar_compra = YES  <- R5
  tendencia_saludable = YES  <- R1
    precio_sobre_ema200 = YES  <- user
    soporte_tecnico_clave = YES  <- user
  oportunidad_compra = YES  <- R3
    rsi_sobreventa = YES  <- user
    volumen_creciente = YES  <- user

CÓMO — Escenario 2 (compra por R1 + R4):
recomendar_compra = YES  <- R5
  tendencia_saludable = YES  <- R1
    precio_sobre_ema200 = YES  <- user
    soporte_tecnico_clave = YES  <- user
  oportunidad_compra = YES  <- R4
    rsi_sobreventa = YES  <- user
    soporte_tecnico_clave = YES  <- user

CÓMO — Escenario 3 (caja por R2 → R6a):
recomendar_caja = YES  <- R6a
  alerta_correccion = YES  <- R2
    rsi_sobrecompra = YES  <- user
    noticias_macro_negativas = YES  <- user


In [12]:
print("POR QUÉ — preguntas del backward hacia recomendar_compra:\n")
for simbolo in ["precio_sobre_ema200", "soporte_tecnico_clave", "rsi_sobreventa", "volumen_creciente"]:
    print("-", why(kb, A("recomendar_compra"), A(simbolo)))

print("\n-", why(kb, A("recomendar_compra"), A("rsi_sobrecompra")))   # caso sin relación con la meta

POR QUÉ — preguntas del backward hacia recomendar_compra:

- Pregunto «el precio está por encima de la EMA de 200 periodos» porque es necesaria para R1 (concluye tendencia_saludable) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
- Pregunto «el precio rebotó en un soporte histórico importante» porque es necesaria para R1 (concluye tendencia_saludable) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
- Pregunto «el RSI está por debajo de 30» porque es necesaria para R3 (concluye oportunidad_compra) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.
- Pregunto «el volumen de las últimas sesiones va en aumento» porque es necesaria para R3 (concluye oportunidad_compra) -> R5 (concluye recomendar_compra), que lleva a la meta recomendar_compra.

- «el RSI está por encima de 70» no interviene en ninguna regla que lleve a «abrir posición de compra (riesgo moderado/bajo)».


## 8. Limitación: conflicto de metas
Con noticias macro negativas **y** condiciones técnicas favorables se pueden deducir `recomendar_compra` y
`recomendar_caja` a la vez, porque Horn no permite expresar «`recomendar_compra` solo si **no** hay noticias negativas».

In [13]:
fb5 = nuevo_caso({
    "precio_sobre_ema200": YES, "soporte_tecnico_clave": YES,
    "rsi_sobreventa": YES, "volumen_creciente": YES,
    "noticias_macro_negativas": YES,
})
forward(kb, fb5)
mostrar_fb(fb5)

compra = fb5.value(A("recomendar_compra")) is YES
caja = fb5.value(A("recomendar_caja")) is YES
print(f"\nrecomendar_compra={compra}, recomendar_caja={caja}  -> conflicto: {compra and caja}")

  precio_sobre_ema200 = YES (user)
  soporte_tecnico_clave = YES (user)
  rsi_sobreventa = YES (user)
  volumen_creciente = YES (user)
  noticias_macro_negativas = YES (user)
  tendencia_saludable = YES (R1)
  oportunidad_compra = YES (R3)
  recomendar_compra = YES (R5)
  recomendar_caja = YES (R6b)

recomendar_compra=True, recomendar_caja=True  -> conflicto: True


In [14]:
def resolver_conflicto(fb):
    """Política de prioridad: ante riesgo (recomendar_caja) se descarta la compra.

    Devuelve el símbolo de la recomendación final o None si no hay ninguna.
    No modifica la FB: la política vive fuera del motor para no alterar sus reglas.
    """
    if fb.value(A("recomendar_caja")) is YES:
        return "recomendar_caja"
    if fb.value(A("recomendar_compra")) is YES:
        return "recomendar_compra"
    return None

print("Recomendación final escenario 5:", resolver_conflicto(fb5))
print("Recomendación final escenario 1:", resolver_conflicto(fb1))

Recomendación final escenario 5: recomendar_caja
Recomendación final escenario 1: recomendar_compra


## 9. Consola interactiva del SBC (Subsistema de Consulta)
El usuario final no llama funciones: usa el **menú del Subsistema de Consulta**. En esta sesión registra los hechos
del Escenario 2 (volumen `UNKNOWN`), ejecuta la inferencia hacia adelante y pide el **cómo** de la recomendación.

Con `INTERACTIVO = True` (sección 6) se abre el menú real y usted responde; si no, se reproduce la sesión guiada.

In [15]:
consulta = QuerySubsystem(kb, TFactBase())   # misma KB, caso nuevo

sesion = [
    "1", "precio_sobre_ema200", "YES",          # 1) agregar hechos observados
    "1", "soporte_tecnico_clave", "YES",
    "1", "rsi_sobreventa", "YES",
    "1", "volumen_creciente", "UNKNOWN",
    "2",                                        # 2) inferir (forward)
    "4", "recomendar_compra",                   # 4) ¿cómo se concluyó?
    "0",                                        # salir
]
if INTERACTIVO:
    consulta.run()
else:
    with respuestas(sesion):
        consulta.run()



[Consulta] 1) Agregar hecho  2) Inferir (forward)  3) Responder meta (backward)  4) Cómo  5) Mostrar FB  6) Reiniciar  0) Salir
Opción: 1
Símbolo del hecho: precio_sobre_ema200
Valor (YES/NO/UNKNOWN): YES

[Consulta] 1) Agregar hecho  2) Inferir (forward)  3) Responder meta (backward)  4) Cómo  5) Mostrar FB  6) Reiniciar  0) Salir
Opción: 1
Símbolo del hecho: soporte_tecnico_clave
Valor (YES/NO/UNKNOWN): YES

[Consulta] 1) Agregar hecho  2) Inferir (forward)  3) Responder meta (backward)  4) Cómo  5) Mostrar FB  6) Reiniciar  0) Salir
Opción: 1
Símbolo del hecho: rsi_sobreventa
Valor (YES/NO/UNKNOWN): YES

[Consulta] 1) Agregar hecho  2) Inferir (forward)  3) Responder meta (backward)  4) Cómo  5) Mostrar FB  6) Reiniciar  0) Salir
Opción: 1
Símbolo del hecho: volumen_creciente
Valor (YES/NO/UNKNOWN): UNKNOWN

[Consulta] 1) Agregar hecho  2) Inferir (forward)  3) Responder meta (backward)  4) Cómo  5) Mostrar FB  6) Reiniciar  0) Salir
Opción: 2
  tendencia_saludable = YES (R1)
  opo

## 10. Conclusiones
**Ventajas de la IA simbólica en finanzas**
- *Transparencia y trazabilidad:* `explain` y `why` justifican cada conclusión y cada pregunta, algo exigible en decisiones de riesgo.
- *Manejo de incertidumbre:* la lógica trivalente y la redundancia R3/R4 permiten decidir con datos incompletos.
- *Mantenibilidad:* el conocimiento (KB) está separado del motor; un experto puede modificarlo sin programar el motor.

**Limitaciones**
- Umbrales fijos (RSI 30/70) y variables booleanas sin grados de certeza.
- Conflicto de metas (escenario 5), mitigado con `resolver_conflicto` fuera del motor.
- Sin negación en las reglas de Horn: `NO` solo bloquea reglas.

**Trabajo futuro:** factores de certeza o lógica difusa, parametrización de umbrales por activo y más indicadores
(MACD, Bandas de Bollinger, métricas on-chain).

## Referencias
- Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4.ª ed.), cap. 7.
- Laboratorio 05 · IAIA 2026-2.